<a href="https://colab.research.google.com/github/Eafit-Ingenieria-Agronomica/Agricultura-Predictiva-2026_I/blob/main/Sesión_06_SeleccionyExtraccionCaracteristicas/PCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

#**Agricultura predictiva**
###**Análisis de Componentes Principales PCA**

In [1]:
#Se importan las librerias necesarias
import os                                                 # to set current working directory
from sklearn.decomposition import PCA                     # PCA program from scikit learn (package for machine learning)
from sklearn.preprocessing import StandardScaler          # standardize variables to mean of 0.0 and variance of 1.0
import pandas as pd                                       # DataFrames and plotting
import pandas.plotting as pd_plot                         # matrix scatter plots
import numpy as np                                        # arrays and matrix math
import matplotlib.pyplot as plt                           # plotting
import seaborn as sns
import ipywidgets as widgets
from ipywidgets import interactive, interact              # widgets and interactivity
from ipywidgets import widgets
from ipywidgets import Layout
from ipywidgets import Label
import matplotlib.transforms as transforms
import math
from ipywidgets import VBox, HBox
import warnings
warnings.filterwarnings('ignore')
cmap = plt.cm.inferno

## **Descripción del dataset**

[El dataset Wine](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_wine.html?utm_source=chatgpt.com) de scikit-learn es un conjunto clásico de aprendizaje automático utilizado para problemas de clasificación multiclase. Contiene 178 muestras de vino provenientes de tres variedades cultivadas en Italia, descritas mediante 13 características químicas y físicas.
El objetivo es predecir la clase de vino a partir de estas variables.



##Se importa el dataset de sklearn

In [2]:
from sklearn.datasets import load_wine
X, y = load_wine(return_X_y=True, as_frame=True)
df = X

##**Exploración de datos**

In [5]:
#Obtener shape de los datos (filas,columnas)


In [ ]:
#Lista de los nombres de los campos de una



In [ ]:
#Mostrar los primeros registros del dataframe


In [ ]:
#Mostrar los útimos registros del dataframe


In [ ]:
#Descripción de la Información de la tabla y tipo de datos


In [ ]:
#Conteo de valores únicos por columna


In [ ]:
# Explorar registros duplicados
df.duplicated().any() #True si hay valores duplicados, False si no

In [ ]:
# Resumen de estadísticas


## **Percentiles: expresado en porcentaje (0 a 100)**
## **Cuantiles: proporción entre 0 y 1**

In [ ]:
# Percentil 2 y 98 usando numpy
numpy_data=np.arange(101)
print("datos =",numpy_data)
p2, p98 = np.percentile(numpy_data, (2, 98))
print("percentil(2)={}".format(p2))
print("percentil(98)={}".format(p98))

In [ ]:
import numpy as np
# Percentil 2 y 98 usando numpy
numpy_data2=np.random.choice(np.arange(255),size=200,replace=True)
print("datos =",numpy_data2)
sorted2=np.sort(numpy_data2)
print("datos ordenados =",sorted2)
p2, p98 = np.percentile(numpy_data2, (2, 98))
print("percentil(2)={}".format(p2))
print("percentil(98)={}".format(p98))

In [ ]:
# Exploración de datos atípicos
Q1 = df.quantile(0.25)
Q3 = df.quantile(0.75)
IQR = Q3 - Q1

outliers_iqr = ((df < (Q1 - 1.5 * IQR)) | (df > (Q3 + 1.5 * IQR)))
outlier_counts_iqr = outliers_iqr.sum()
print("\nNúmero de outliers detectados por variable (IQR):")
print(outlier_counts_iqr)

In [ ]:
#Imputación de los datos atípicos con el método de Winsor (cambia por los límites máximo y mínimo)
df_winsor = df.copy()
for col in df.columns:
    low = Q1[col] - 1.5 * IQR[col]
    high = Q3[col] + 1.5 * IQR[col]
    df_winsor[col] = np.where(df_winsor[col] < low, low,
                       np.where(df_winsor[col] > high, high, df_winsor[col]))

In [ ]:
# Transformar datos con el método standardScaler
scaler = StandardScaler() #Se usa este porque es más común para el análisis de PCA
df_scaled = pd.DataFrame(scaler.fit_transform(df_winsor),
                         columns=df.columns, index=df.index)

In [ ]:
# Boxplots de las variables (antes y después de transformar)

# Antes (con outliers)
plt.figure(figsize=(12,6))
df.boxplot(rot=45)
plt.title("Boxplot de todas las variables (original)")
plt.tight_layout()
plt.show()

# Después (outliers imputados)
Q1s, Q3s = df_scaled.quantile(0.25), df_scaled.quantile(0.75)
IQRs = Q3s - Q1s
df_scaled_winsor = df_scaled.clip(lower=Q1s - 1.5*IQRs, upper=Q3s + 1.5*IQRs, axis=1)
plt.figure(figsize=(12,6))
df_scaled_winsor.boxplot(rot=45)
plt.title("Boxplot de todas las variables (transformadas, outliers imputados)")
plt.tight_layout()
plt.show()

##**Análisis de correlación con gráficos**

In [ ]:
#Análisis de correlación entre las variables
df= df_scaled
plt.figure(figsize=(12,8))
sns.heatmap(df.corr(), annot=True, cmap=cmap, fmt=".2f")
plt.show()

In [ ]:
#Scatter plot
pd_plot = __import__("pandas").plotting
scatter = pd_plot.scatter_matrix(df_scaled, alpha=0.1, figsize=(10,10),
                                 color='black', hist_kwds={'color':['grey']})

# Ajusta etiquetas
for ax in scatter.ravel():
    ax.set_xlabel(ax.get_xlabel(), rotation=45, ha='right')
    ax.set_ylabel(ax.get_ylabel(), rotation=0, ha='right')

plt.suptitle("Matrix Scatter Plot")
#plt.tight_layout()
plt.show()

## **Análisis de componentes principales (PCA)**

## Concepto de **varianza**

En probabilidad, la varianza de una variable aleatoria
𝑋 es una medida de cuánto varían, en promedio, los valores de la distribución con respecto a la media.

La varianza Var[X] se calcula como el promedio de los cuadrados de las diferencias entre cada valor de la distribución y el valor promedio de la distribución .

Var[𝑋] sera igual al promedio de [(𝑋−promedio[𝑋])^2]

En NumPy, la varianza puede calcularse para un vector o una matriz usando la función var(). Por defecto, la función var() calcula la varianza poblacional. Para calcular la varianza muestral, se debe establecer el argumento ddof con el valor 1.

El siguiente ejemplo define un vector y calcula la varianza muestral

In [ ]:
v = np.array([2,4,6])
print("valores variable = ",v)

#calcular varianza usando la funcion np.var()
varianza = np.var(v, ddof=1)# varianza muestral

print("varianza variable =",varianza)

#calcular varianza con operaciones basicas
prom=np.mean(v)#Hallar la media
dev=np.square(v-prom)#Restar la media y elevar al cuadrado para garantizar que el resultado sea positivo
print("desviaciones de la media =",dev)
n=dev.size#obtener la cantidad de items en el arreglo
sample_var=np.sum(dev)/(n-1)#sumar desviaciones y dividir por n-1
print("varianza muestral de la variable =", sample_var)

##Concepto de **covarianza**

La covarianza, como concepto base, está definida **entre pares de variables**. Es una medida que describe **cómo dos variables varían conjuntamente**.

- Si ambas variables tienden a aumentar o disminuir juntas, la covarianza es positiva.

- Si una variable aumenta mientras la otra disminuye, la covarianza es negativa.

- Si no tienen una relación lineal clara, la covarianza es cercana a cero.

Por definición la covarianza solo involucra dos variables, cuando se tiene un conjunto de más de dos variables, como:

𝑋1,𝑋2,𝑋3,…,𝑋𝑛

se calcula la covarianza entre cada par de variables para todas las combinaciones de pares de variables posibles, y se organiza todo en una matriz de covarianza

La [matriz de covarianza](https://es.wikipedia.org/wiki/Matriz_de_covarianza#:~:text=En%20estad%C3%ADstica%20y%20teor%C3%ADa%20de,de%20una%20variable%20aleatoria%20escalar.) es una matriz cuadrada y [simétrica](https://es.wikipedia.org/wiki/Matriz_sim%C3%A9trica#:~:text=Una%20matriz%20es%20sim%C3%A9trica%20si,matrices%20cuadradas%20pueden%20ser%20sim%C3%A9tricas) que describe la covarianza para todas las posibles combinaciones de pares de variables presentes en los datos.

La diagonal de la matriz de covarianza contiene las varianzas de cada una de las variables esto sucede porque **la covarianza de una variable consigo misma** cov(X,X) **es igual a la varianza de la variable**.

Una matriz de covarianza es una generalización de la covarianza entre dos variables y captura la manera en que todas las variables del conjunto de datos pueden cambiar conjuntamente.

La matriz de covarianza se denota con la letra griega mayúscula Sigma (Σ). La covarianza para cada par de variables se calcula como se mostró anteriormente.

![](https://wikimedia.org/api/rest_v1/media/math/render/svg/5f2e0b96d940a309a42591dbeca4ffcdc490be1d)

La matriz de correlación es en esencia una versión normalizada de la matriz de covarianza, donde se normaliza cada elemento dividiéndolo entre el producto de las desviaciones estándar del par de variables asociadas a cada elemento de la matriz.

Sus valores representan un índice de correlación (r) que puede variar entre -1 y +1, los extremos del rango indicando correlaciones perfectas, negativa y positiva respectivamente.
- Un valor de r = 0 indica que no existe relación lineal entre las dos variables.
- Una correlación positiva indica que ambas variables varían en el mismo sentido.
- Una correlación negativa significa que ambas variables varían en sentidos opuestos.

Lo interesante de este índice de correlación es que **aporta información tanto de la variación (+-) además de una medida de la fuerza del efecto**. Esta matriz presenta unos en su diagonal principal que representan la correlación de la cada variable con ella misma

In [ ]:
# datos de las variables
x1 = np.array([2, 4, 6])
x2 = np.array([5, 9, 13])

#creamos una tabla donde cada columna contiene los datos de cada variable
table=pd.DataFrame({"v1":x1,"v2":x2})
table

In [ ]:
#Hallar matriz de covarianza usando numpy (no es necesario centrar los datos, la funcion np.cov() lo hace automáticamente durante el cálculo)
cov_matrix = np.cov(table.to_numpy(), ddof=1, rowvar=False)# rowvar=False abliga a que se tomen a las columnas como las variables
print("\ncov_matrix:")
print(cov_matrix)

In [ ]:
#Hallar matriz de correlacion usando numpy, recomendado cuando existen diferencias de escala entre variables,
#además su salida esta -1 y +1 de facil interpretacion aportando informacion tanto de la fuerza como de la variación(+-) conjunta
corr_matrix =np.corrcoef(table.to_numpy(), rowvar=False)# rowvar=False abliga a que se tomen a las columnas como las variables
print("\ncov_matrix:")
print(corr_matrix)

In [ ]:
#Seleccionemos una de las matrices anteriores a modo de experimentacion
selected_matrix=cov_matrix

Comentario: la función **np.cov()** puede ser útil para aprender cómo funciona PCA desde cero, **pero su sensibilidad ante la escala de las variables limita su aplicación**.

## Valores y vectores propios

In [ ]:
#Calcular valores propios Vals y vectores propios P de la matriz de covarianza

#np.linalg.eigh devuelve los vectores propios ordenados por valores propios de menor a mayor
#requiere reordenar de mayor a menor con respecto a valores propios para escoger los valores propios

Vals, P = np.linalg.eigh(selected_matrix)

print("\nvalores propios")
print(Vals)
print("\nvectores propios:")
print(P)
print("cada columna es el vector propio asociado a cada valor propio")
varianza_explicada=[vp/np.sum(Vals) for vp in Vals]
print("\nvarianza explicada =",np.round(varianza_explicada,3))
#calcular el indice(posicion) asociado al valor maximo en el arreglo de valores propios
indice_maximo=np.argmax(Vals)
print("indice del mayor valor propio =", indice_maximo)

$\text{Varianza explicada por cada componente } = \frac{\lambda_i}{\sum_{j=1}^{n} \lambda_j}
\quad \text{donde } \lambda_i \text{ es el valor propio asociado al componente } i
$


In [ ]:
#Transformemos los datos multiplicando los datos centrados, por el vector propio con mayor valor propio asociado
datos_centrados=(table-table.mean()).to_numpy()
#Seleccionar de forma manual el vector propio para este caso particular usamos [0.4472136,0.89442719], cambiar cuando sea necesario
vector_propio_maximo=np.array([0.4472136,0.89442719]).reshape(-1,1) # reshape para que la salida sea un vector columna
# el operador @ en numpy se usa para multiplicación matricial o se puede usar tambien np.dot()
pca=np.dot(datos_centrados,vector_propio_maximo)
print("coordenadas proyectadas sobre el vector propio:")
print(pca)

## Datos del vino y PCA usando funciones nativas de numpy

In [ ]:
#Esto suprime la notación científica del print de numpy, para ver mejor el cambio en las cifras al usar la matriz de covarianza o de correlacion
np.set_printoptions(suppress=True, precision=5)# usar suppress=False para volver al comportamiento por defecto

In [ ]:
# Hallar matriz de covarianza usando numpy
cov_matrix = np.cov(df_scaled.to_numpy(), ddof=1, rowvar=False) #varianza muestral, covarianza por columnas
print("\ncov_matrix:")
print(cov_matrix)


In [ ]:
#Calcular valores propios "Vals" y vectores propios P ​​de la matriz de covarianza
Vals, P = np.linalg.eigh(cov_matrix)
print("\nvalores propios")
print(Vals)
print("\nvectores propios:")
print(P)
print("cada columna es el vector propio asociado a cada valor propio")

## Componente principal

In [ ]:
# Centrar los datos
datos_centrados = df_scaled.to_numpy() # df_scale ya esta estandarizado

# Tomar el autovector asociado al mayor autovalor
idx_max = np.argmax(Vals)        # índice del valor propio más grande
vector_propio_maximo = P[:, idx_max].reshape(-1,1)  # columna del vector propio

# Proyección de los datos
pca = datos_centrados @ vector_propio_maximo

print("coordenadas proyectadas sobre el vector propio:")
print(pca)

## Usando sklearn

In [ ]:
# Datos
data = df_scaled.to_numpy() # df ya esta estandarizada

#Centrar o estandarizar los datos
#scaler = StandardScaler()
# Aplicar PCA usando 2 componente principal
pca = PCA(n_components=2)
principal_components = pca.fit_transform(data)

print("Original data shape:", data.shape)
print("Transformed data shape:", principal_components.shape)
print("Explained Variance Ratio:", pca.explained_variance_ratio_)
print("Principal Components (Eigenvectors):\n", pca.components_)
print("Transformed data (principal components):\n", principal_components)

In [ ]:
# Otra forma de hacerlo
# 1. Ajustar PCA (tantos componentes como variables)
pca = PCA()
X_pca = pca.fit_transform(df_scaled)

# 2. Varianza explicada por cada componente
var_exp = pca.explained_variance_ratio_
cum_var_exp = var_exp.cumsum()

# 3. DataFrame resumen
pca_summary = pd.DataFrame({
    "Componente": [f"PC{i+1}" for i in range(len(var_exp))],
    "Varianza explicada": var_exp,
    "Varianza acumulada": cum_var_exp
})
print(pca_summary)

# 4. Gráfico Scree plot (método del codo)
plt.figure(figsize=(7,5))
plt.plot(range(1, len(var_exp)+1), cum_var_exp, marker='o')
plt.xticks(range(1, len(var_exp)+1), [f"PC{i}" for i in range(1, len(var_exp)+1)], rotation=45)
plt.xlabel("Componente principal")
plt.ylabel("Varianza explicada acumulada")
plt.title("Método del codo (PCA)")
plt.grid(True)
plt.tight_layout()
plt.show()

De forma arbitraria se eligen 7 componentes principales, debido a que representan casi el 90% de la varianza, corresponde con la inflexión del codo y la diferencia en varianza de este PC al siguiente es muy baja

El resultado del método de numpy y de sklearn son iguales, solo que sklearn reduce las líneas de código necesarias para analizar los componentes principales.



Las variables total_phenols y flavanoids tienen alta correlación de 0.89, se hará un ejemplo con ellas

In [ ]:
def firstvariable(feature1="total_phenols", feature2="flavanoids"):
    column1 = feature1; column2 = feature2
    my_data_sel = df[[column1,column2]].iloc[0:100]   # tomar primeras 100 filas

    print("1: Visual Distribution")
    f, (ax1, ax2) = plt.subplots(1, 2, sharey=True, figsize=(8,3))
    ax1.hist(my_data_sel[column1], alpha=0.2, color='red', edgecolor='black', bins=20)
    ax1.set_title(column1)
    ax2.hist(my_data_sel[column2], alpha=0.2, color='red', edgecolor='black', bins=20)
    ax2.set_title(column2)
    plt.show()

    # Escalado
    features = [column1, column2]
    x = my_data_sel.loc[:,features].values
    mu = np.mean(x, axis=0)
    sd = np.std(x, axis=0)
    x = StandardScaler().fit_transform(x)

    print("2: Original and Transformed Statistics\n")
    print(f"Original Mean {column1} = {np.round(mu[0],2)}, Original Mean {column2} = {np.round(mu[1],2)}")
    print(f"Original StDev {column1} = {np.round(sd[0],2)}, Original StDev {column2} = {np.round(sd[1],2)}")
    print(f"Mean Transformed {column1} = {np.round(np.mean(x[:,0]),2)}, Mean Transformed {column2} = {np.round(np.mean(x[:,1]),2)}")
    print(f"Variance Transformed {column1} = {np.round(np.var(x[:,0]),2)}, Variance Transformed {column2} = {np.round(np.var(x[:,1]),2)}")

    # PCA con 2 componentes
    n_components = 2
    pca = PCA(n_components=n_components)
    pca.fit(x)
    print("\n3: Covariance and Variance Explained Matrices\n")
    print(np.round(pca.components_,3))
    print("Variance explained by PC1 and PC2 =", np.round(pca.explained_variance_ratio_,3))

    print("\n4: Original and PCA Cross-plots")
    f, (ax101, ax102, ax103) = plt.subplots(1, 3, figsize=(12,3))
    f.subplots_adjust(wspace=0.7)

    # Original standardized scatter
    ax101.scatter(x[:,0], x[:,1], c="red", alpha=0.3, edgecolors="black")
    ax101.set_title(f'Standardized {column2} vs. {column1}')
    ax101.set_xlabel(f'Standardized {column1}')
    ax101.set_ylabel(f'Standardized {column2}')

    # PCA transform scatter
    x_trans = pca.transform(x)
    ax102.scatter(x_trans[:,0], x_trans[:,1], c="red", alpha=0.3, edgecolors="black")
    ax102.set_title('Principal Component Scores')
    ax102.set_xlabel('PC1')
    ax102.set_ylabel('PC2')

    # Reverse PCA scatter
    x_reverse = pca.inverse_transform(x_trans)
    ax103.scatter(x_reverse[:,0], x_reverse[:,1], c="red", alpha=0.3, edgecolors="black")
    ax103.set_title('Reverse PCA')
    ax103.set_xlabel(f'Standardized {column1}')
    ax103.set_ylabel(f'Standardized {column2}')

    plt.show()

    # Varianzas comparativas



# Llamada a la función para tus variables de interés
firstvariable("total_phenols", "flavanoids")